In [3]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke('top news in india today')

print(results)

Hindustan Times: Stay updated with Hindustan Times for the top India news, world events, and breaking stories. Get real-time updates on politics, wars, live cricket scores, entertainment news, and ... India News | Latest India News | Read latest and breaking news from India. Today's top India news headlines, news on Indian politics, elections, government, business, technology, and Bollywood. Stay updated with the latest breaking news, live updates, photos, videos and top trending stories from India and around the world across politics, economy, sports and more on The Economic Times. Times of India brings the Latest News & Breaking News Headlines from India & around the World. Read Latest News Today on Sports, Business, Health & Fitness, Bollywood & Entertainment, Blogs ... The latest news headlines on Life Style, Business, Politics, Hollywood and News from across the world at The Times of India.


## Built-in Tool - Shell Tool

In [7]:
from langchain_experimental.tools import ShellTool

shell_tool = ShellTool()

result = shell_tool.invoke({"commands": ["ls"]})

print(result)

ImportError: cannot import name 'ShellTool' from 'langchain_experimental.tools' (/home/bs/gen-ai/langchain-campusX/venv/lib/python3.13/site-packages/langchain_experimental/tools/__init__.py)

## Custom Tools

In [8]:
 #Step 1 - create a function

def multiply(a, b):
    """Multiply two numbers"""
    return a*b

In [9]:
# Step 2 - add type hints

def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [11]:
# Step 3 - add tool decorator
from langchain_core.tools import tool

@tool
def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [12]:
result = multiply.invoke({"a":3, "b":5})

In [13]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [14]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


## Method 2 - Using StructuredTool

A StructuredTool in LangChain is a special type of tool where the input to the tool follows a structured schema, typically defined using a Pydantic schema.


In [17]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [18]:
class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

/tmp/ipykernel_446027/3279971488.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True, description="The first number to add")
/tmp/ipykernel_446027/3279971488.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True, description="The second number to add")


In [19]:
def multiply_func(a: int, b: int) -> int:
    return a * b

In [20]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

In [21]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


## Method 3 - Using BaseTool Class

---

BaseTool is an abstract base class in LangChain that defines the common interface and behavior that all tools should follow.

In simple words:

BaseTool = blueprint/interface for creating LangChain tools.

StructuredTool, @tool, and other tools ultimately follow the tool interface defined by BaseTool.


              BaseTool
                 ↓
        ┌────────┼────────┐
        ↓        ↓        ↓
 StructuredTool  Tool   Other Tools

BaseTool khud usually actual kaam nahi karta.
Ye bas batata hai ki har LangChain tool mein basic cheezein kaise honi chahiye.

For example, tool ke paas:

name
description
input lene ka mechanism
invoke() karne ka mechanism hona chye


In [25]:
from langchain_core.tools import BaseTool
from pydantic import BaseModel, Field
from typing import Type

In [23]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

/tmp/ipykernel_446027/908171234.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True, description="The first number to add")
/tmp/ipykernel_446027/908171234.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True, description="The second number to add")


In [26]:
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

In [28]:
multiply_tool = MultiplyTool()
result = multiply_tool.invoke({'a':3, 'b':3})
print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


![Image](image.png)

In [30]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

In [31]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]

In [32]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)


add => Add two numbers
multiply => Multiply two numbers


In [33]:

add_tool = tools[0]
multiply_tool = tools[1]

print(add_tool.invoke({"a": 10, "b": 20}))
print(multiply_tool.invoke({"a": 10, "b": 20}))

30
200
